# 03 · Geração fundamentada: JSON válido, citação e recusa

Modelo local: **qwen2.5:3b** via Ollama (temperatura 0, semente 42). Saída estruturada com JSON Schema do próprio Ollama e validação com **pydantic** no formato do contrato:
```json
{"resposta": str, "fontes": [{"documento": str, "trecho_id": str}], "confianca": "alta|media|baixa", "nao_consta": bool}
```
Regras de validação: confiança só com 3 valores; resposta afirmativa precisa citar pelo menos uma fonte; recusa não cita fonte.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
import pandas as pd
pd.set_option('display.max_colwidth', 120)

In [2]:
from pf_genai import gerador, llm
from pf_genai.esquemas import RespostaIA
from pf_genai.indice import Indice
idx = Indice.carregar()
print('LLM:', llm.MODELO, '| limiar:', gerador.LIMIAR)

LLM: qwen2.5:3b | limiar: 0.86


In [3]:
def mostrar(r):
    print(f'[{r.configuracao}] {r.latencia_ms} ms | JSON válido: {r.json_valido} | recuperados: {list(zip(r.recuperados, r.scores))}')
    print(json.dumps(r.resposta.model_dump() if r.resposta else {'erro': r.erro}, ensure_ascii=False, indent=2))

mostrar(gerador.rag('Tenho 67 anos. Preciso pagar a passagem do ônibus em São Paulo?', idx))

[rag] 4903 ms | JSON válido: True | recuperados: [('sptrans-gratuidade-65-mais-008', 0.8895), ('sptrans-gratuidade-65-mais-001', 0.8855), ('sptrans-gratuidade-65-mais-028', 0.8832)]
{
  "resposta": "Não, você não precisa pagar a passagem do ônibus. Você tem direito à gratuidade com o Bilhete Único Especial da Pessoa Idosa.",
  "fontes": [
    {
      "documento": "sptrans-gratuidade-65-mais-008",
      "trecho_id": "sptrans-gratuidade-65-mais-008"
    },
    {
      "documento": "sptrans-gratuidade-65-mais-001",
      "trecho_id": "sptrans-gratuidade-65-mais-001"
    }
  ],
  "confianca": "alta",
  "nao_consta": false
}


## Recusa quando não há base
Perguntas fora da base são barradas pelo limiar (sem chamar o LLM) ou pela instrução de recusa.

In [4]:
mostrar(gerador.rag('Qual o horário do último ônibus da linha 875C-10?', idx))
mostrar(gerador.rag('Vai chover amanhã em São Paulo?', idx))

[rag] 60 ms | JSON válido: True | recuperados: [('sptrans-bilhete-pcd-033', 0.806), ('sptrans-gratuidade-65-mais-013', 0.8056), ('sptrans-bilhete-pcd-014', 0.8029)]
{
  "resposta": "Não encontrei essa informação nas fontes oficiais que eu conheço. Para ter certeza, ligue para a SPTrans no telefone 156.",
  "fontes": [],
  "confianca": "media",
  "nao_consta": true
}
[rag] 54 ms | JSON válido: True | recuperados: [('sptrans-gratuidade-60-64-014', 0.8061), ('sptrans-bilhete-unico-031', 0.8042), ('sptrans-bilhete-pcd-002', 0.7986)]
{
  "resposta": "Não encontrei essa informação nas fontes oficiais que eu conheço. Para ter certeza, ligue para a SPTrans no telefone 156.",
  "fontes": [],
  "confianca": "media",
  "nao_consta": true
}


## Validação pydantic barrando saídas fora do contrato

In [5]:
for bruto in [{'resposta': 'É grátis.', 'fontes': [], 'confianca': 'alta', 'nao_consta': False},
              {'resposta': 'É grátis.', 'fontes': [{'documento': 'Lei', 'trecho_id': 'x'}], 'confianca': 'certeza', 'nao_consta': False}]:
    print(gerador._validar(bruto)[1])

Value error, resposta sem nao_consta precisa citar ao menos uma fonte
Input should be 'alta', 'media' or 'baixa'


## O mesmo modelo sem contexto
Para comparação: sem a base, o modelo responde com o que "lembra" do treino, e cita fontes que não existem.

In [6]:
mostrar(gerador.sem_contexto('Tenho 67 anos. Preciso pagar a passagem do ônibus em São Paulo?'))
mostrar(gerador.sem_contexto('Qual o horário do último ônibus da linha 875C-10?'))

[sem_contexto] 1163 ms | JSON válido: True | recuperados: []
{
  "resposta": "Não, você não precisa pagar a passagem. Você pode usar o Bilhete Único, que é gratuito para idosos a partir de 60 anos.",
  "fontes": [],
  "confianca": "alta",
  "nao_consta": true
}


[sem_contexto] 812 ms | JSON válido: True | recuperados: []
{
  "resposta": "O horário do último ônibus da linha 875C-10 é às 23:00. ",
  "fontes": [],
  "confianca": "alta",
  "nao_consta": true
}
